In [5]:
from Bio import motifs, SeqIO
from Bio.Seq import Seq
from collections import Counter
import re

# 1. Load sequences from FASTA file
sequences = [str(record.seq) for record in SeqIO.parse("E:\\master\\final_project\\data\\phage_scope\\all.fasta", "fasta")]

print(f"Loaded {len(sequences)} sequences")
print(f"Sequence lengths: {[len(seq) for seq in sequences[:5]]}")  # Show first 5 lengths

# === SOLUTION 1: EXTRACT MOTIFS OF FIXED LENGTH ===
print("\n=== PHƯƠNG PHÁP 1: TẠO MOTIF TỪ CÁC ĐOẠN CÓ ĐỘ DÀI BẰNG NHAU ===")

def extract_fixed_length_subsequences(sequences, motif_length=10):
    """Extract subsequences of fixed length from all sequences"""
    fixed_length_seqs = []

    for seq in sequences:
        if len(seq) >= motif_length:
            # Extract from beginning of sequence
            fixed_length_seqs.append(seq[:motif_length])

            # Or extract from middle if you prefer
            # start_pos = len(seq) // 2 - motif_length // 2
            # fixed_length_seqs.append(seq[start_pos:start_pos + motif_length])

    return fixed_length_seqs

# Extract 10-nucleotide subsequences
motif_length = 10
fixed_seqs = extract_fixed_length_subsequences(sequences, motif_length)
print(f"Extracted {len(fixed_seqs)} subsequences of length {motif_length}")

# Convert to Seq objects
bio_seqs = [Seq(seq) for seq in fixed_seqs]

# Create motif
try:
    m1 = motifs.create(bio_seqs)
    print(f"Motif created successfully!")
    print(f"Consensus sequence: {m1.consensus}")
    print(f"Position Weight Matrix shape: {len(m1.pwm)} x {len(m1.pwm[0])}")
except Exception as e:
    print(f"Error creating motif: {e}")

# === SOLUTION 2: FIND COMMON MOTIFS USING SLIDING WINDOW ===
print("\n=== PHƯƠNG PHÁP 2: TÌM MOTIF PHỔ BIẾN BẰNG SLIDING WINDOW ===")

def find_common_kmers(sequences, k=6, min_occurrences=3):
    """Find k-mers that appear frequently across sequences"""
    kmer_counts = Counter()

    for seq in sequences:
        # Extract all k-mers from this sequence
        kmers_in_seq = set()  # Use set to count each k-mer once per sequence
        for i in range(len(seq) - k + 1):
            kmer = seq[i:i+k]
            if 'N' not in kmer:  # Skip k-mers with ambiguous nucleotides
                kmers_in_seq.add(kmer)

        # Add to global counter
        for kmer in kmers_in_seq:
            kmer_counts[kmer] += 1

    # Return k-mers that appear in at least min_occurrences sequences
    common_kmers = {kmer: count for kmer, count in kmer_counts.items()
                   if count >= min_occurrences}

    return common_kmers

# Find common 6-mers
common_kmers = find_common_kmers(sequences, k=6, min_occurrences=5)
print(f"Found {len(common_kmers)} common 6-mers")

# Show top 10 most common
sorted_kmers = sorted(common_kmers.items(), key=lambda x: x[1], reverse=True)
print("Top 10 most common 6-mers:")
for kmer, count in sorted_kmers[:10]:
    print(f"  {kmer}: appears in {count} sequences")

# === SOLUTION 3: CREATE MOTIF FROM SPECIFIC PATTERN ===
print("\n=== PHƯƠNG PHÁP 3: TẠO MOTIF TỪ MẪU CỤ THỂ ===")

def find_pattern_instances(sequences, pattern="TGAC", context_length=8):
    """Find instances of a specific pattern with surrounding context"""
    instances = []

    for seq in sequences:
        for match in re.finditer(pattern, seq, re.IGNORECASE):
            start = max(0, match.start() - context_length//2)
            end = min(len(seq), match.end() + context_length//2)

            instance = seq[start:end]
            if len(instance) == context_length + len(pattern):
                instances.append(instance)

    return instances

# Find instances of "TGAC" with context
pattern_instances = find_pattern_instances(sequences, "TGAC", context_length=6)
print(f"Found {len(pattern_instances)} instances of 'TGAC' with context")

if pattern_instances:
    # Create motif from pattern instances
    pattern_seqs = [Seq(instance) for instance in pattern_instances[:100]]  # Limit to first 100

    try:
        m3 = motifs.create(pattern_seqs)
        print(f"Pattern-based motif created!")
        print(f"Consensus sequence: {m3.consensus}")

        # Create sequence logo (if you have logomaker installed)
        # import logomaker
        # logo_df = pd.DataFrame(m3.pwm)
        # logomaker.Logo(logo_df)

    except Exception as e:
        print(f"Error creating pattern-based motif: {e}")

# === SOLUTION 4: ANALYZE SEQUENCE STATISTICS ===
print("\n=== PHƯƠNG PHÁP 4: PHÂN TÍCH THỐNG KÊ CÁC CHUỖI ===")

def analyze_sequences(sequences):
    """Analyze basic statistics of sequences"""
    lengths = [len(seq) for seq in sequences]

    print(f"Total sequences: {len(sequences)}")
    print(f"Length range: {min(lengths)} - {max(lengths)}")
    print(f"Average length: {sum(lengths)/len(lengths):.1f}")
    print(f"Median length: {sorted(lengths)[len(lengths)//2]}")

    # Nucleotide composition
    all_nucs = ''.join(sequences)
    nuc_counts = Counter(all_nucs)
    total_nucs = sum(nuc_counts.values())

    print("Nucleotide composition:")
    for nuc in 'ATCG':
        count = nuc_counts.get(nuc, 0)
        percentage = (count / total_nucs) * 100 if total_nucs > 0 else 0
        print(f"  {nuc}: {count} ({percentage:.1f}%)")

analyze_sequences(sequences)

print("\n=== RECOMMENDATIONS ===")
print("1. Use Solution 1 if you want to analyze motifs from sequence starts/ends")
print("2. Use Solution 2 to find naturally occurring common motifs")
print("3. Use Solution 3 if you're looking for specific patterns")
print("4. Consider filtering sequences by length before motif analysis")
print("5. Install 'logomaker' package to visualize motifs: pip install logomaker")

Loaded 6722 sequences
Sequence lengths: [4532, 31754, 28451, 134416, 49136]

=== PHƯƠNG PHÁP 1: TẠO MOTIF TỪ CÁC ĐOẠN CÓ ĐỘ DÀI BẰNG NHAU ===
Extracted 6722 subsequences of length 10
Motif created successfully!
Consensus sequence: ATGGGATTTT
Position Weight Matrix shape: 4 x 10

=== PHƯƠNG PHÁP 2: TÌM MOTIF PHỔ BIẾN BẰNG SLIDING WINDOW ===
Found 4114 common 6-mers
Top 10 most common 6-mers:
  GATGAA: appears in 6685 sequences
  TGGTGG: appears in 6682 sequences
  TGATGA: appears in 6679 sequences
  AAGATG: appears in 6679 sequences
  ATGGTG: appears in 6677 sequences
  ATGGCT: appears in 6675 sequences
  CAAGGT: appears in 6674 sequences
  CAAGAA: appears in 6672 sequences
  TTCAAG: appears in 6669 sequences
  GTTGCT: appears in 6669 sequences

=== PHƯƠNG PHÁP 3: TẠO MOTIF TỪ MẪU CỤ THỂ ===
Found 1890329 instances of 'TGAC' with context
Pattern-based motif created!
Consensus sequence: AATTGACGGA

=== PHƯƠNG PHÁP 4: PHÂN TÍCH THỐNG KÊ CÁC CHUỖI ===
Total sequences: 6722
Length range: 20

In [4]:
from Bio import motifs, SeqIO
from Bio.Seq import Seq

# 1. Cài đặt: pip install biopython

# 2. Tạo dữ liệu mẫu - các chuỗi DNA có chứa motif "TGAC"
sequences = [str(record.seq) for record in SeqIO.parse("E:\master\\final_project\data\phage_scope\\all.fasta", "fasta")]

# Chuyển đổi sang đối tượng Seq
seqs = [Seq(seq) for seq in sequences]

# 3. Tạo motif từ các chuỗi có sẵn
print("=== PHƯƠNG PHÁP 1: TẠO MOTIF TỪ CÁC CHUỖI CÓ SẴN ===")
m = motifs.create(seqs)

print("Consensus sequence:", m.consensus)
print("Ma trận đếm (Count Matrix):")
print(m.counts)

# 4. Tính ma trận tần suất và PWM
print("\n=== MA TRẬN POSITION WEIGHT MATRIX (PWM) ===")
pwm = m.counts.normalize(pseudocounts=0.5)
pssm = pwm.log_odds()

print("Ma trận tần suất:")
print(pwm)
print("\nPosition Specific Scoring Matrix (PSSM):")
print(pssm)

# 5. Tìm kiếm motif trong chuỗi mới
print("\n=== TÌM KIẾM MOTIF TRONG CHUỖI MỚI ===")
test_sequence = Seq("CCATGACGTTAAATGACCCGGG")
print(f"Chuỗi test: {test_sequence}")

# Quét chuỗi để tìm motif
threshold = 0.0  # Ngưỡng điểm số
for position, score in pssm.search(test_sequence, threshold=threshold):
    print(f"Vị trí {position}: điểm số = {score:.2f}")
    print(f"  Motif tìm thấy: {test_sequence[position:position + len(m)]}")

# 6. Ví dụ khám phá motif đơn giản từ nhiều chuỗi
print("\n=== KHÁM PHÁ MOTIF ĐỐI VỚI NHIỀU CHUỖI ===")


def find_common_kmers(sequences, k=4):
    """Tìm k-mer xuất hiện phổ biến trong các chuỗi"""
    kmer_count = {}

    for seq in sequences:
        # Tạo tất cả k-mer từ chuỗi
        for i in range(len(seq) - k + 1):
            kmer = seq[i:i + k]
            kmer_count[kmer] = kmer_count.get(kmer, 0) + 1

    # Sắp xếp theo tần suất
    sorted_kmers = sorted(kmer_count.items(), key=lambda x: x[1], reverse=True)
    return sorted_kmers

print("Các k-mer phổ biến nhất (k=4):")
common_kmers = find_common_kmers(sequences, k=4)
for kmer, count in common_kmers[:5]:  # Top 5
    print(f"  {kmer}: xuất hiện {count} lần")

# 7. Tạo logo motif (cần matplotlib)
print("\n=== TẠO MOTIF LOGO ===")
try:
    import matplotlib.pyplot as plt
    from Bio import motifs

    # Tạo WebLogo-style representation
    print("Để tạo motif logo, bạn có thể sử dụng:")
    print("1. weblogo: pip install weblogo")
    print("2. logomaker: pip install logomaker")
    print("3. Hoặc xuất ra định dạng JASPAR để sử dụng với các công cụ khác")

    # Xuất định dạng JASPAR
    print("\nĐịnh dạng JASPAR của motif:")
    print(m.format("jaspar"))

except ImportError:
    print("Cần cài matplotlib để tạo logo: pip install matplotlib")

print("\n=== HƯỚNG DẪN SỬ DỤNG ===")
print("1. Cài đặt: pip install biopython")
print("2. Để phân tích nâng cao: pip install scikit-bio")
print("3. Để tạo logo: pip install weblogo logomaker")
print("4. Để sử dụng MEME Suite: cài đặt MEME Suite và dùng subprocess để gọi")

=== PHƯƠNG PHÁP 1: TẠO MOTIF TỪ CÁC CHUỖI CÓ SẴN ===


ValueError: sequences must have the same length if coordinates is None